In [ ]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv()

In [ ]:
llm = ChatOpenAI()

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseModel],add_messages]

In [ ]:
def chat_node(state:ChatState):

    # take user query from state
    messages = state['messages']
    # send to llm
    response = llm.invoke(messages)
    # response store state
    return {'messages':[response]}

In [ ]:
checkpoint = MemorySaver()

graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node',chat_node)

# add edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
chatbot

In [ ]:
thread_id = "2"
initial_state = {
    'messages':[HumanMessage(content='what is my name?')]
}
config = {'configurable':{'thread_id':thread_id}}
response = chatbot.invoke(initial_state,config=config)

In [ ]:
response['messages'][-1].content

In [ ]:
chatbot.get_state(config=config)

In [ ]:
thread_id = "1"

while True:
    user_message = input('typed here: ')

    print('user:',user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break

    config = {'configurable':{'thread_id':thread_id}}
    response = chatbot.invoke({'message':[HumanMessage(content=user_message)]},config=config)
    print('AI: ',response['messages'][-1].content)